# G. NumPy for Sampled Signals

A sampled ECG fits naturally into a NumPy array: one number per sample, in time order. For multiple channels, I use rows for samples and columns for channels. Checking shape, dtype, and units before calculating prevents many quiet mistakes.

## G.1 Create a signal and time axis

In [1]:
import numpy as np
fs = 4
x = np.array([0.0, 0.2, 1.0, 0.1, -0.1, 0.2, 0.8, 0.0], dtype=float)
t = np.arange(x.size) / fs
print("Shape:", x.shape)
print("Time (s):", t)
print("Values (mV):", x)

Shape: (8,)
Time (s): [0.   0.25 0.5  0.75 1.   1.25 1.5  1.75]
Values (mV): [ 0.   0.2  1.   0.1 -0.1  0.2  0.8  0. ]


This is a teaching waveform, not a realistic ECG. The sampling interval is 0.25 s and the last sample time is 1.75 s. The segment contains eight sampling intervals when interpreted as a two-second sampled segment.

## G.2 Indices, slices, and masks

Indices start at zero. `x[start:stop]` includes `start` and excludes `stop`. Boolean masks select positions satisfying a condition.

In [2]:
print("First sample:", x[0])
print("Samples 2 through 4:", x[2:5])
print("Samples in [0.5, 1.5) s:", x[(t >= 0.5) & (t < 1.5)])
print("Values above 0.5 mV:", x[x > 0.5])

First sample: 0.0
Samples 2 through 4: [ 1.   0.1 -0.1]
Samples in [0.5, 1.5) s: [ 1.   0.1 -0.1  0.2]
Values above 0.5 mV: [1.  0.8]


## G.3 Shapes, axes, and broadcasting

In an array of shape `(samples, channels)`, averaging along `axis=0` returns one mean per channel. `keepdims=True` preserves a row shape so those values can be subtracted from every row by broadcasting.

In [3]:
signals = np.column_stack([x, 0.5 * x + 0.1])
means = signals.mean(axis=0, keepdims=True)
centered = signals - means
print("Signal shape:", signals.shape)
print("Mean shape:", means.shape)
print("Means (mV):", means)
print("Centered means:", np.round(centered.mean(axis=0), 12))

Signal shape: (8, 2)
Mean shape: (1, 2)
Means (mV): [[0.275  0.2375]]
Centered means: [-0. -0.]


## G.4 Arithmetic and RMS

`*` multiplies corresponding array elements; `@` performs matrix multiplication. RMS is the square root of the average squared value, so its units match the input. Here the RMS includes any DC component.

In [4]:
rms = np.sqrt(np.mean(x ** 2))
print("RMS (mV):", round(float(rms), 4))
print("Successive differences:", np.diff(x))
print("Difference length:", np.diff(x).size)
assert np.isclose(rms ** 2, np.mean(x * x))

RMS (mV): 0.4664
Successive differences: [ 0.2  0.8 -0.9 -0.2  0.3  0.6 -0.8]
Difference length: 7


## G.5 Views, copies, and saving arrays

A basic slice usually shares memory with its source. Use `.copy()` when editing a segment independently.

In [5]:
view = x[:3]
independent = x[:3].copy()
print("Slice shares memory:", np.shares_memory(view, x))
print("Copy shares memory:", np.shares_memory(independent, x))
independent[0] = 99
print("Original first sample:", x[0])

Slice shares memory: True
Copy shares memory: False
Original first sample: 0.0


`.npy` files preserve array shape and dtype. A named `.npz` archive can hold an array alongside its sampling rate. Record units and channel meaning as well; those are not implied by a numerical dtype.

Selecting `x[::2]` does not apply an anti-aliasing filter. Use an appropriate resampling method before reducing a real recording's sampling rate.

## Further reading

- [NumPy beginner guide](https://numpy.org/doc/stable/user/absolute_beginners.html)
- [Copies and views](https://numpy.org/doc/stable/user/basics.copies.html)
- [Aliasing and anti-aliasing](../ch09_sampling/index.md)